# 01. Preprocessing — RGB→CMYK Conversion, ROI Extraction, Patch Save
# 01. 전처리 — RGB→CMYK 변환, ROI 추출(고정좌표), 패치 저장

**Goal / 목표**
- Convert RGB scan image to CMYK 4 channels
- RGB 스캔 이미지를 CMYK 4채널로 변환한다
- Extract each channel ROI using fixed coordinates
- 고정 좌표를 사용하여 각 채널 ROI를 추출한다

**Folder Structure / 폴더 구조**
```
data/
├── images/          ← 원본 스캔 이미지 / Raw scan images
└── labeled/
    ├── C/ 0/ 1/ 2/ 3/ 4/ 5/
    ├── M/ 0/ 1/ 2/ 3/ 4/ 5/
    ├── Y/ 0/ 1/ 2/ 3/ 4/ 5/
    └── K/ 0/ 1/ 2/ 3/ 4/ 5/
```

---
## 1. Import Libraries / 라이브러리 임포트

In [ ]:
import os
import random
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset
from pathlib import Path

print('Libraries loaded / 라이브러리 로드 완료')

---
## 2. Path & Settings / 경로 및 설정

> config.yaml 없이 직접 경로를 지정한다.
> Paths are set directly without config.yaml.

In [ ]:
# 프로젝트 루트 기준 경로 / Paths based on project root
ROOT_DIR    = Path('../..').resolve()           # CMYK_MAIN/
RAW_DIR     = ROOT_DIR / 'data_set' / 'images'  # 원본 스캔 이미지 / Raw scan images
LABELED_DIR = ROOT_DIR / 'data_set' / 'labeled' # 라벨링 폴더 / Labeled folder

CHANNELS   = ['Y', 'M', 'C', 'K']
NUM_LEVELS = 6       # Level 0 ~ 5
IMAGE_SIZE = 128     #  기준 크기 /  standard size

# 고정 ROI 좌표 (x, y, w, h) / Fixed ROI coordinates
# 실제 스캔 이미지 크기에 맞게 조정 필요 / Adjust to actual scan image dimensions
ROI_COORDS = {
    'Y': [0,   0,   800, 200],
    'M': [0,   200, 800, 200],
    'C': [0,   400, 800, 200],
    'K': [0,   600, 800, 200],
}

print(f'Root    : {ROOT_DIR}')
print(f'Raw dir : {RAW_DIR}')
print(f'Labeled : {LABELED_DIR}')

# 라벨링 폴더 샘플 수 확인 / Check labeled folder sample counts
print('\nLabeled folder counts / 라벨링 폴더 샘플 수:')
for ch in CHANNELS:
    for lv in range(NUM_LEVELS):
        folder = LABELED_DIR / ch / str(lv)
        count  = len(list(folder.glob('*'))) if folder.exists() else 0
        print(f'  [{ch}] Level {lv}: {count}개')

---
## 3. RGB → CMYK Conversion / RGB → CMYK 변환

In [ ]:
def rgb_to_cmyk(rgb: np.ndarray) -> dict:
    """
    RGB 이미지를 CMYK 4채널로 변환한다.
    Converts an RGB image to CMYK 4 channels.

    Args:
        rgb: (H, W, 3) uint8 이미지 / uint8 image

    Returns:
        dict: {"Y": ..., "M": ..., "C": ..., "K": ...} each (H, W) float32 [0,1]
    """
    rgb_f = rgb.astype(np.float32) / 255.0
    R, G, B = rgb_f[:, :, 0], rgb_f[:, :, 1], rgb_f[:, :, 2]

    # K 채널 계산 / Compute K channel
    K = 1.0 - np.maximum(np.maximum(R, G), B)

    # 0 나눗셈 방지 / Prevent division by zero
    denom = np.where((1.0 - K) < 1e-6, 1e-6, 1.0 - K)

    C = (1.0 - R - K) / denom
    M = (1.0 - G - K) / denom
    Y = (1.0 - B - K) / denom

    return {
        'Y': np.clip(Y, 0, 1),
        'M': np.clip(M, 0, 1),
        'C': np.clip(C, 0, 1),
        'K': np.clip(K, 0, 1),
    }


# 변환 테스트 / Conversion test
test_rgb  = np.array([[[255, 0, 0]]], dtype=np.uint8)  # 순수 빨강 / Pure red
test_cmyk = rgb_to_cmyk(test_rgb)
print('RGB → CMYK 변환 테스트 / Conversion test (Pure Red [255,0,0]):')
for ch, val in test_cmyk.items():
    print(f'  [{ch}]: {val[0,0]:.3f}')

---
## 4. ROI Extraction (Fixed Coordinates) / ROI 추출 (고정 좌표)

In [ ]:
def extract_roi_fixed(cmyk: dict, coords: dict) -> dict:
    """
    고정 좌표(x, y, w, h)로 CMYK 채널별 ROI를 추출한다.
    Extracts per-channel ROI using fixed coordinates (x, y, w, h).
    """
    rois = {}
    for ch in CHANNELS:
        x, y, w, h = coords[ch]
        rois[ch]   = cmyk[ch][y:y+h, x:x+w].copy()
    return rois


# 실제 이미지로 테스트 / Test with real image
exts   = {'.png', '.jpg', '.jpeg', '.tiff', '.tif'}
images = [p for p in RAW_DIR.rglob('*') if p.suffix.lower() in exts]

if not images:
    print('[WARN] No images found / 이미지 없음')
    print(f'       Path checked / 확인 경로: {RAW_DIR}')
else:
    sample_path = images[0]
    print(f'Image found / 이미지 발견: {sample_path.name}')

    img  = cv2.imread(str(sample_path))
    rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    cmyk = rgb_to_cmyk(rgb)
    rois = extract_roi_fixed(cmyk, ROI_COORDS)

    print('ROI extraction results / 추출 결과:')
    for ch, roi in rois.items():
        print(f'  [{ch}] shape: {roi.shape} | range: [{roi.min():.3f}, {roi.max():.3f}]')

---
## 5. Preprocessing / Preprocessing

> `preprocessing.py` 기준 — resize 128×128 + normalize
> Based on `preprocessing.py` — resize 128×128 + normalize

In [ ]:
def preprocess(image: np.ndarray) -> np.ndarray:
    """
     전처리 — resize + normalize.
     standard preprocessing — resize + normalize.

    Args:
        image: (H, W, 3) uint8 또는 float

    Returns:
        (IMAGE_SIZE, IMAGE_SIZE, 3) float32 [0,1]
    """
    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    image = image / 255.0
    return image.astype(np.float32)


# 전처리 테스트 / Preprocessing test
if images:
    patches = {}
    for ch, roi in rois.items():
        roi_u8      = (roi * 255).astype(np.uint8)
        # 단일채널 → 3채널 복제 / Replicate to 3 channels
        roi_3ch     = np.stack([roi_u8, roi_u8, roi_u8], axis=-1)
        patches[ch] = preprocess(roi_3ch)

    print('Preprocessing results / 전처리 결과:')
    for ch, patch in patches.items():
        print(f'  [{ch}] shape: {patch.shape} | range: [{patch.min():.3f}, {patch.max():.3f}]')

---
## 6. Augmentation (R1 기준) / Augmentation

In [ ]:
def augment(image: np.ndarray) -> np.ndarray:
    """
     기준 증강 — flip, brightness, noise.
     standard augmentation — flip, brightness, noise.

    Args:
        image: (H, W, 3) uint8

    Returns:
        Augmented image (same shape)
    """
    # 수평 뒤집기 / Horizontal flip
    if random.random() > 0.5:
        image = cv2.flip(image, 1)

    # 밝기 변동 / Brightness jitter
    if random.random() > 0.5:
        value = random.randint(-30, 30)
        image = cv2.add(image, value)

    # 노이즈 추가 / Add noise
    if random.random() > 0.5:
        noise = random.randint(0, 10)
        image = image + noise

    return image


# 증강 테스트 / Augmentation test
if images:
    sample_u8 = (patches['Y'] * 255).astype(np.uint8)
    augmented = augment(sample_u8.copy())
    print('Augmentation test / 증강 테스트:')
    print(f'  Original  shape: {sample_u8.shape}')
    print(f'  Augmented shape: {augmented.shape}')

---
## 7. Dataset Class / Dataset Class

**폴더 구조 / Folder structure:**
```
data/labeled/{channel}/{level}/*.png
예시 / Example: data/labeled/C/2/scan_001_C_0007.png
```

In [ ]:
class CMYKDataset(Dataset):
    """
     기준 Dataset — data_set/labeled/{channel}/{level}/ 폴더 구조 기반.
     standard Dataset — based on data_set/labeled/{channel}/{level}/ folder structure.
    """

    def __init__(self, channel: str, transform=None):
        """
        Args:
            channel:   "Y" | "M" | "C" | "K"
            transform: torchvision transform (선택 / optional)
        """
        self.channel   = channel
        self.transform = transform
        self.samples   = []  # [(image_path, level), ...]
        self.exts      = {'.png', '.jpg', '.jpeg', '.tiff', '.tif'}

        # data_set/labeled/{channel}/{level}/ 에서 샘플 수집
        # Collect samples from data_set/labeled/{channel}/{level}/
        channel_dir = LABELED_DIR / channel
        for level in range(NUM_LEVELS):
            level_dir = channel_dir / str(level)  # 0, 1, 2, 3, 4, 5
            if not level_dir.exists():
                continue
            for img_path in sorted(level_dir.glob('*')):
                if img_path.suffix.lower() in self.exts:
                    self.samples.append((img_path, level))

        print(f'[{channel}] Dataset loaded / 로드 완료: {len(self.samples)}개 샘플 / samples')

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img_path, level = self.samples[idx]

        # 이미지 로드 / Load image
        image = cv2.imread(str(img_path))
        if image is None:
            raise ValueError(f'Image not found / 이미지 없음: {img_path}')

        #  기준 전처리 /  standard preprocessing
        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
        image = image / 255.0

        # (H, W, 3) → (3, H, W) 텐서 변환 / Convert to tensor
        image = torch.tensor(image).permute(2, 0, 1).float()

        if self.transform:
            image = self.transform(image)

        return image, level


# Dataset 로드 테스트 / Dataset load test
for ch in CHANNELS:
    ds = CMYKDataset(channel=ch)
    if len(ds) > 0:
        x, y = ds[0]
        print(f'  [{ch}] Sample shape: {x.shape} | Level: {y}')